# LLM Fine-Tuning for Verilog/RTL Generation with Unsloth

**Goal**: Fine-tune a Qwen3.5 or Llama model on Verilog code generation using Unsloth + QLoRA

**GPU**: Tesla T4 (14.6 GB VRAM)

**Method**: QLoRA 4-bit quantization for memory efficiency

## Part 1: Install Unsloth

In [3]:
# Configure compiler/toolchain BEFORE importing torch or Unsloth
import os
import shutil
import subprocess
import sys
from pathlib import Path

os.environ.setdefault('TRITON_CACHE_DIR', '/tmp/triton-cache')
os.environ.setdefault('TORCHINDUCTOR_CACHE_DIR', '/tmp/torchinductor-cache')

compiler_candidates = [
    ('/tmp/conda-envs/gcc_env/bin/gcc', '/tmp/conda-envs/gcc_env/bin/g++'),
    ('/tmp/conda-envs/gcc_env/bin/x86_64-conda-linux-gnu-gcc', '/tmp/conda-envs/gcc_env/bin/x86_64-conda-linux-gnu-g++'),
    (shutil.which('gcc'), shutil.which('g++')),
    (shutil.which('x86_64-conda-linux-gnu-gcc'), shutil.which('x86_64-conda-linux-gnu-g++')),
]

selected_cc = None
selected_cxx = None
for cc, cxx in compiler_candidates:
    if cc and cxx and Path(cc).exists() and Path(cxx).exists():
        selected_cc = str(Path(cc).resolve())
        selected_cxx = str(Path(cxx).resolve())
        break

RTL_NOTEBOOK_COMPILER_READY = selected_cc is not None and selected_cxx is not None
RTL_NOTEBOOK_SELECTED_CC = selected_cc
RTL_NOTEBOOK_SELECTED_CXX = selected_cxx

if RTL_NOTEBOOK_COMPILER_READY:
    compiler_bin = str(Path(selected_cc).parent)
    os.environ['PATH'] = compiler_bin + os.pathsep + os.environ.get('PATH', '')
    os.environ['CC'] = selected_cc
    os.environ['CXX'] = selected_cxx
    print(f"Using C compiler: {os.environ['CC']}")
    print(f"Using C++ compiler: {os.environ['CXX']}")
    print(subprocess.check_output([os.environ['CC'], '--version'], text=True).splitlines()[0])
else:
    os.environ['TORCH_COMPILE'] = '0'
    os.environ['UNSLOTH_COMPILE_DISABLE'] = '1'
    os.environ['TRITON_COMPILE_DISABLE'] = '1'
    os.environ['VLLM_DISABLE_COMPILE_CACHE'] = '1'
    print('No compiler detected; using compile-disabled fallback mode.')

import torch
torch._dynamo.config.suppress_errors = True
torch._inductor.config.fallback_random = True

print('Installing Unsloth...')
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--user', '-q',
                       'unsloth', 'trl', 'peft', 'accelerate'])
print('✓ Unsloth installed!')


Using C compiler: /usr/local/bin/x86_64-conda-linux-gnu-gcc
Using C++ compiler: /usr/local/bin/x86_64-conda-linux-gnu-g++
x86_64-conda-linux-gnu-gcc (Anaconda gcc) 11.2.0
Installing Unsloth...
✓ Unsloth installed!


In [4]:
# Verify compiler configuration
import os
import shutil
import subprocess

print(f"Compiler ready: {RTL_NOTEBOOK_COMPILER_READY}")
print(f"CC: {os.environ.get('CC', 'unset')}")
print(f"CXX: {os.environ.get('CXX', 'unset')}")
print(f"Selected compiler: {globals().get('RTL_NOTEBOOK_SELECTED_CC', 'unset')}")
print(f"Selected C++ compiler: {globals().get('RTL_NOTEBOOK_SELECTED_CXX', 'unset')}")
print(f"gcc on PATH: {shutil.which('gcc')}")
print(f"g++ on PATH: {shutil.which('g++')}")
if RTL_NOTEBOOK_COMPILER_READY:
    cc = os.environ['CC']
    cxx = os.environ['CXX']
    print(subprocess.check_output([cc, '--version'], text=True).splitlines()[0])
    print(subprocess.check_output([cxx, '--version'], text=True).splitlines()[0])
else:
    print('Compiler not available; notebook will stay in fallback mode.')


Compiler ready: True
CC: /usr/local/bin/x86_64-conda-linux-gnu-gcc
CXX: /usr/local/bin/x86_64-conda-linux-gnu-g++
Selected compiler: /usr/local/bin/x86_64-conda-linux-gnu-gcc
Selected C++ compiler: /usr/local/bin/x86_64-conda-linux-gnu-g++
gcc on PATH: None
g++ on PATH: None
x86_64-conda-linux-gnu-gcc (Anaconda gcc) 11.2.0
x86_64-conda-linux-gnu-g++ (Anaconda gcc) 11.2.0


In [5]:
# Verify GPU availability
import torch

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

PyTorch: 2.11.0+cu130
CUDA available: True
GPU: Tesla T4
VRAM: 14.6 GB


## Part 2: Download Verilog Datasets

In [6]:
# Download Verilog datasets from Hugging Face
from datasets import load_dataset
from pathlib import Path

dataset_dir = Path.cwd() / 'datasets' / 'llm'
dataset_dir.mkdir(parents=True, exist_ok=True)

print("Downloading Verilog datasets...")
print("="*60)

In [7]:
# Dataset 1: LLM_4_Verilog (15K samples, structured instructions)
print("\n1. Loading LLM_4_Verilog (15K samples)...")
try:
    llm4verilog = load_dataset("NOKHAB-Lab/LLM_4_Verilog", split="train")
    print(f"   ✓ Loaded {len(llm4verilog)} samples")
    print(f"   Columns: {llm4verilog.column_names}")
    
    # Save to JSONL
    llm4verilog.to_json(dataset_dir / 'llm_4_verilog.jsonl')
    print(f"   ✓ Saved to: {dataset_dir / 'llm_4_verilog.jsonl'}")
except Exception as e:
    print(f"   ✗ Error: {e}")
    llm4verilog = None


1. Loading LLM_4_Verilog (15K samples)...


README.md: 0.00B [00:00, ?B/s]

   ✗ Error: Dataset 'NOKHAB-Lab/LLM_4_Verilog' is a gated dataset on the Hub. You must be authenticated to access it.


In [8]:
# Dataset 2: PyraNet-Verilog (692K samples, largest)
print("\n2. Loading PyraNet-Verilog (692K samples)...")
try:
    pyranet = load_dataset("bnadimi/PyraNet-Verilog", split="train")
    print(f"   ✓ Loaded {len(pyranet)} samples")
    print(f"   Columns: {pyranet.column_names}")
    
    # Sample 10K for faster training
    pyranet_sample = pyranet.shuffle(seed=42).select(range(10000))
    pyranet_sample.to_json(dataset_dir / 'pyranet_verilog_10k.jsonl')
    print(f"   ✓ Saved 10K sample to: {dataset_dir / 'pyranet_verilog_10k.jsonl'}")
except Exception as e:
    print(f"   ✗ Error: {e}")
    pyranet = None


2. Loading PyraNet-Verilog (692K samples)...


README.md: 0.00B [00:00, ?B/s]

PyraNetOnVeriBest.csv:   0%|          | 0.00/2.68G [00:00<?, ?B/s]

Generating train split:   0%|          | 0/692238 [00:00<?, ? examples/s]

   ✓ Loaded 692238 samples
   Columns: ['code', 'description']


Creating json from Arrow format:   0%|          | 0/10 [00:00<?, ?ba/s]

   ✓ Saved 10K sample to: /home/jovyan/silicogen/rtl_analyzer/notebooks/rtl_analyzer_phase3/datasets/llm/pyranet_verilog_10k.jsonl


In [9]:
# Dataset 3: Verilog_GitHub (109K samples from VeriGen paper)
print("\n3. Loading Verilog_GitHub (109K samples)...")
try:
    verilog_gh = load_dataset("shailja/Verilog_GitHub", split="train")
    print(f"   ✓ Loaded {len(verilog_gh)} samples")
    
    verilog_gh.to_json(dataset_dir / 'verilog_github.jsonl')
    print(f"   ✓ Saved to: {dataset_dir / 'verilog_github.jsonl'}")
except Exception as e:
    print(f"   ✗ Error: {e}")
    verilog_gh = None


3. Loading Verilog_GitHub (109K samples)...


README.md: 0.00B [00:00, ?B/s]

Verilog_bigquery_GitHub.csv:   0%|          | 0.00/1.89G [00:00<?, ?B/s]

Generating train split:   0%|          | 0/108971 [00:00<?, ? examples/s]

   ✓ Loaded 108971 samples


Creating json from Arrow format:   0%|          | 0/109 [00:00<?, ?ba/s]

   ✓ Saved to: /home/jovyan/silicogen/rtl_analyzer/notebooks/rtl_analyzer_phase3/datasets/llm/verilog_github.jsonl


In [10]:
# Summary
print("\n" + "="*60)
print("DATASET SUMMARY")
print("="*60)

files = list(dataset_dir.glob('*.jsonl'))
for f in files:
    size_mb = f.stat().st_size / 1024 / 1024
    print(f"{f.name}: {size_mb:.1f} MB")

print(f"\nTotal datasets: {len(files)}")
print(f"Directory: {dataset_dir}")


DATASET SUMMARY
pyranet_verilog_10k.jsonl: 41.4 MB
verilog_github.jsonl: 1906.0 MB
verilog_combined.jsonl: 204.1 MB

Total datasets: 3
Directory: /home/jovyan/silicogen/rtl_analyzer/notebooks/rtl_analyzer_phase3/datasets/llm


In [11]:
# Preview a sample
if llm4verilog:
    print("\nSample from LLM_4_Verilog:")
    print("="*60)
    sample = llm4verilog[0]
    print(f"Module: {sample.get('idx', 'N/A')}")
    print(f"Circuit Type: {sample.get('circuit_type', 'N/A')}")
    if 'code' in sample:
        print(f"\nCode (first 200 chars):\n{sample['code'][:200]}...")
    if 'instruction' in sample and isinstance(sample['instruction'], dict):
        print(f"\nDescription:\n{sample['instruction'].get('description', 'N/A')[:200]}...")

## Part 3: Prepare Dataset for Training

In [12]:
# Format dataset for instruction tuning
from datasets import concatenate_datasets

def format_for_training(example):
    """Convert to instruction-response format"""
    code = example.get('code', '')
    
    # Try different field names
    instruction = (
        example.get('instruction', '') or
        example.get('description', '') or
        example.get('text', '') or
        f"Generate Verilog code for module {example.get('module_name', 'design')}"
    )
    
    if isinstance(instruction, dict):
        instruction = instruction.get('description', str(instruction))
    
    return {
        'instruction': instruction[:500],  # Truncate long instructions
        'output': code,
        'text': f"""### Instruction:
Generate Verilog code for: {instruction[:500]}

### Response:
{code}"""
    }

print("Formatting datasets...")

Formatting datasets...


In [13]:
# Process and combine datasets
all_datasets = []

for name, ds in [('LLM_4_Verilog', llm4verilog), ('PyraNet', pyranet_sample if pyranet else None), ('Verilog_GitHub', verilog_gh)]:
    if ds:
        print(f"Processing {name}...")
        formatted = ds.map(format_for_training, remove_columns=ds.column_names)
        all_datasets.append(formatted)
        print(f"  ✓ {len(formatted)} samples")

if all_datasets:
    combined = concatenate_datasets(all_datasets)
    print(f"\n✓ Combined dataset: {len(combined)} samples")
    
    # Save combined
    combined.to_json(dataset_dir / 'verilog_combined.jsonl')
    print(f"✓ Saved to: {dataset_dir / 'verilog_combined.jsonl'}")
else:
    print("No datasets loaded!")
    combined = None

Processing PyraNet...


Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

  ✓ 10000 samples
Processing Verilog_GitHub...


Map:   0%|          | 0/108971 [00:00<?, ? examples/s]

  ✓ 108971 samples

✓ Combined dataset: 118971 samples


Creating json from Arrow format:   0%|          | 0/119 [00:00<?, ?ba/s]

✓ Saved to: /home/jovyan/silicogen/rtl_analyzer/notebooks/rtl_analyzer_phase3/datasets/llm/verilog_combined.jsonl


## Part 4: Fine-Tune with Unsloth + QLoRA

In [14]:
# Check if we have data to train
if combined is None or len(combined) == 0:
    print("No training data available. Run previous cells first.")
else:
    print(f"Ready to train on {len(combined)} samples!")

Ready to train on 118971 samples!


In [15]:
# Keep compile-disabling only as a fallback for no-compiler environments
if not globals().get('RTL_NOTEBOOK_COMPILER_READY', False):
    torch.compiler.disable(recursive=True)
    print('Compiler unavailable: keeping torch.compile disabled for model loading.')
else:
    print(f"Compiler available for Triton builds: {os.environ.get('CC', 'gcc')}")

# Load model with Unsloth (Qwen3.5-4B recommended for T4)
from unsloth import FastLanguageModel
import torch

# For Tesla T4: Use 4-bit quantization (QLoRA)
max_seq_length = 1024  # Reduce if OOM
load_in_4bit = True

print(f"Loading model (max_seq={max_seq_length}, 4bit={load_in_4bit})...")

# Choose model based on VRAM:
# - Qwen3.5-0.8B: ~3GB VRAM (safest)
# - Qwen3.5-2B: ~5GB VRAM (recommended)
# - Qwen3.5-4B: ~7GB VRAM (best quality)

model_name = 'unsloth/Qwen3.5-2B'  # Best balance for T4

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=max_seq_length,
    load_in_4bit=load_in_4bit,
    load_in_16bit=False,
    full_finetuning=False,
)

print(f"✓ Model loaded: {model_name}")


Compiler available for Triton builds: /usr/local/bin/x86_64-conda-linux-gnu-gcc
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
Loading model (max_seq=1024, 4bit=True)...
==((====))==  Unsloth 2026.3.11: Fast Qwen3_5 patching. Transformers: 5.3.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu130. CUDA: 7.5. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Using float16 precision for qwen3_5 won't work! Using float32.


model.safetensors.index.json: 0.00B [00:00, ?B/s]

model.safetensors-00001-of-00001.safeten(…):   0%|          | 0.00/4.55G [00:00<?, ?B/s]

The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


Loading weights:   0%|          | 0/617 [00:00<?, ?it/s]

...

preprocessor_config.json:   0%|          | 0.00/336 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/20.0M [00:00<?, ?B/s]

video_preprocessor_config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

✓ Model loaded: unsloth/Qwen3.5-2B


In [16]:
# Configure LoRA adapters
model = FastLanguageModel.get_peft_model(
    model,
    r=8,  # Lower rank for T4 (use 16 if you have VRAM)
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=8,
    lora_dropout=0,  # Unsloth optimized
    bias="none",
    use_gradient_checkpointing="unsloth",  # Critical for long sequences
    random_state=3407,
    max_seq_length=max_seq_length,
)

print(f"✓ LoRA adapters configured")
print(f"  Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

Unsloth: Making `model.base_model.model.model.language_model` require gradients
✓ LoRA adapters configured
  Trainable parameters: 5,455,872


In [ ]:
# Training configuration for Tesla T4
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=combined,
    dataset_text_field="text",
    args=SFTConfig(
        # Critical for T4:
        per_device_train_batch_size=1,  # MUST be 1-2
        gradient_accumulation_steps=8,  # Effective batch = 8
        
        # Training schedule:
        max_steps=200,  # Increase for production (500-1000)
        learning_rate=2e-5,
        lr_scheduler_type="cosine",
        warmup_ratio=0.05,
        
        # Optimization:
        optim="adamw_8bit",  # Saves VRAM
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        
        # Logging:
        logging_steps=10,
        output_dir="outputs_verilog_llm",
        save_strategy="steps",
        save_steps=50,
        
        # Reproducibility:
        seed=3407,
        dataset_num_proc=1,
    ),
)

print("✓ Trainer configured")
print(f"  Effective batch size: {1 * 8}")
print(f"  Total training steps: {200}")

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Unsloth: Switching to float32 training since model cannot work with float16


Unsloth: Tokenizing ["text"] (num_proc=1):   0%|          | 0/118971 [00:00<?, ? examples/s]

✓ Trainer configured
  Effective batch size: 8
  Total training steps: 200


: 

In [ ]:
# Train!
print("Starting training...")
print("="*60)

import time
start_time = time.time()

trainer_stats = trainer.train()

elapsed = time.time() - start_time
print("="*60)
print(f"✓ Training complete!")
print(f"  Time: {elapsed/60:.1f} minutes")
print(f"  Final loss: {trainer_stats.training_loss:.4f}")
print(f"  GPU peak memory: {torch.cuda.max_memory_allocated() / 1024**3:.2f} GB")

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 248046}.


Starting training...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 118,971 | Num Epochs = 1 | Total steps = 200
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 8 x 1) = 8
 "-____-"     Trainable parameters = 5,455,872 of 2,218,697,536 (0.25% trained)
/home/jovyan/.local/lib/python3.12/site-packages/bitsandbytes/_ops.py:239: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
/home/jovyan/.local/lib/python3.12/site-packages/bitsandbytes/_ops.py:186: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
/home/jovyan/.local/lib/python3.12/site-packages/bitsandbytes/_ops.py:239: FutureWarning: _check_is_size will be removed in a future P

Step,Training Loss


Unsloth: Will smartly offload gradients to save VRAM!


/home/jovyan/.local/lib/python3.12/site-packages/bitsandbytes/_ops.py:239: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
/home/jovyan/.local/lib/python3.12/site-packages/bitsandbytes/_ops.py:186: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
/home/jovyan/.local/lib/python3.12/site-packages/bitsandbytes/_ops.py:239: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
/home/jovyan/.local/lib/python3.12/site-packages/bitsandbytes/_ops.py:186: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
/home/jo

## Part 5: Save and Test Model

In [ ]:
# Save model
from pathlib import Path

model_dir = Path("verilog_finetuned_model")
model_dir.mkdir(exist_ok=True)

# Option 1: Save LoRA adapter only (~100MB)
model.save_pretrained(model_dir / "lora_adapter")
print(f"✓ LoRA adapter saved: {model_dir / 'lora_adapter'}")

# Option 2: Save merged model for vLLM (~4GB for 2B model)
# model.save_pretrained_merged(model_dir / "merged", tokenizer, save_method="merged_16bit")

# Option 3: Save as GGUF for llama.cpp/Ollama
# model.save_pretrained_gguf(model_dir / "gguf", tokenizer, quantization_method="q4_k_m")

In [ ]:
# Test generation
FastLanguageModel.for_inference(model)

test_prompts = [
    "Generate Verilog code for a 4-bit counter with reset",
    "Create a FIFO buffer with 8-bit data width",
    "Write a simple ALU module with add and subtract operations",
]

print("Testing fine-tuned model:")
print("="*60)

for prompt in test_prompts:
    print(f"\nPrompt: {prompt}")
    print("-"*60)
    
    inputs = tokenizer(f"### Instruction:\n{prompt}\n\n### Response:\n", return_tensors="pt").to("cuda")
    outputs = model.generate(**inputs, max_new_tokens=256, temperature=0.7, top_p=0.9)
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    
    # Extract just the response part
    if "### Response:" in response:
        response = response.split("### Response:")[1].strip()
    
    print(response[:500] + "..." if len(response) > 500 else response)

## Summary

### What We Did:
1. ✓ Installed Unsloth for efficient fine-tuning
2. ✓ Downloaded 3 Verilog datasets (up to 816K samples)
3. ✓ Formatted for instruction tuning
4. ✓ Fine-tuned Qwen3.5-2B with QLoRA on Tesla T4
5. ✓ Saved model and tested generation

### Next Steps:
- Increase training steps (500-1000) for better quality
- Try larger models (Qwen3.5-4B) if VRAM allows
- Evaluate on VerilogEval benchmark
- Deploy with vLLM or Ollama

### Resources:
- Unsloth Docs: https://unsloth.ai/docs
- More datasets: Search "verilog" on Hugging Face
- VerilogEval benchmark: https://github.com/verilog-eval/VerilogEval